In [1]:
from analytics.db import query

In [2]:
query("""
    SELECT 
        COALESCE(t.product_category_name_english, 'unknown') AS category_name,
        COUNT(p.product_id) AS product_count
    FROM products AS p
    LEFT JOIN category_translation AS t 
        ON p.product_category_name = t.product_category_name
    GROUP BY category_name
    ORDER BY product_count DESC
    LIMIT 5
""")

,category_name,product_count
0,bed_bath_table,3029
1,sports_leisure,2867
2,furniture_decor,2657
3,health_beauty,2444
4,housewares,2335


### Part 0: Correction on Orders & Customers Relationship
* **Correct Relationship:** One-to-One (1:1) between `orders` and `customers`.
* **Why the previous guess was wrong:** In Olist's data architecture, the `customers` table does not store unique lifetime customers as a parent entity; instead, each row represents a unique customer-order instance (`customer_id`), resulting in a direct one-to-one match with the `orders` table.

In [3]:
query("""
    SELECT COUNT(*)
    FROM orders AS o
    JOIN customers AS c ON c.customer_id = o.customer_id
""")

,count
0,99441


In [4]:
query("""
    SELECT COUNT(*)
    FROM orders AS o
    JOIN order_items AS i ON i.order_id = o.order_id
""")

,count
0,112650


In [5]:
query("""
    SELECT COUNT(*)
    FROM customers AS c
    JOIN geolocation AS g ON g.geolocation_zip_code_prefix = c.customer_zip_code_prefix
""")

,count
0,15083455


In [6]:
query("""
    SELECT
        (SELECT COUNT(*) FROM customers   WHERE customer_zip_code_prefix    = '07953') AS customer_rows,
        (SELECT COUNT(*) FROM geolocation WHERE geolocation_zip_code_prefix = '07953') AS geolocation_rows

""")

,customer_rows,geolocation_rows
0,3,5


In [7]:
   query("""
       SELECT COUNT(*)
       FROM customers AS c
       JOIN geolocation AS g ON g.geolocation_zip_code_prefix = c.customer_zip_code_prefix
       WHERE c.customer_zip_code_prefix = '07953'
   """)

,count
0,15


In [8]:
   query("""
       SELECT SUM(c.n * g.n) AS rows_the_rule_gives
       FROM (
           SELECT customer_zip_code_prefix AS zip, COUNT(*) AS n
           FROM customers
           GROUP BY customer_zip_code_prefix
       ) AS c
       JOIN (
           SELECT geolocation_zip_code_prefix AS zip, COUNT(*) AS n
           FROM geolocation
           GROUP BY geolocation_zip_code_prefix
       ) AS g ON g.zip = c.zip
   """)

,rows_the_rule_gives
0,15083455.0


In [9]:
  query("""
       SELECT COUNT(*) AS rows, COUNT(DISTINCT o.order_id) AS orders
       FROM orders AS o
       JOIN order_items AS i ON i.order_id = o.order_id
   """)

,rows,orders
0,112650,98666


In [10]:
query("""
      -- ex:1.a
    
    SELECT 
        review_count,
        COUNT(*) AS orders_count
    FROM (
        SELECT o.order_id, COUNT(r.review_id) AS review_count
        FROM orders AS o
        LEFT JOIN order_reviews AS r ON r.order_id = o.order_id
        GROUP BY o.order_id
    ) AS sub
    GROUP BY review_count
    ORDER BY review_count DESC
""")

,review_count,orders_count
0,3,4
1,2,543
2,1,98126
3,0,768


> **Interpretation:** Most orders receive exactly 1 review, while a small fraction have multiple or zero reviews.

In [11]:
query("""
    -- ex : b.1  
    
    SELECT COUNT(*) 
    FROM orders AS o
    LEFT JOIN order_items AS i ON i.order_id = o.order_id
""")

,count
0,113425


> **Interpretation (b.1):** Using a LEFT JOIN between orders and order_items yields a row count higher than total orders because individual orders containing multiple items expand across multiple rows.


In [12]:
  query("""
       SELECT order_status, COUNT(*) AS orders
       FROM orders
       GROUP BY order_status
       ORDER BY orders DESC
   """)

,order_status,orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [13]:
query("SELECT ROUND(AVG(price), 2) AS average_price FROM order_items")

,average_price
0,120.65


In [14]:
   query("""
       SELECT ROUND(AVG(items_total), 2) AS average_items_total
       FROM (
           SELECT order_id, SUM(price) AS items_total
           FROM order_items
           GROUP BY order_id
       ) AS per_order
   """)

,average_items_total
0,137.75


In [15]:
   query("""
       WITH per_order AS (
           SELECT order_id, SUM(price) AS items_total
           FROM order_items
           GROUP BY order_id
       )
       SELECT ROUND(AVG(items_total), 2) AS average_items_total
       FROM per_order
   """)

,average_items_total
0,137.75


In [16]:
   query("""
       WITH per_order AS (
           -- one row = one order that has items
           SELECT order_id, SUM(price) AS items_total
           FROM order_items
           GROUP BY order_id
       ),
       per_order_with_state AS (
           -- one row = one order that has items, with the state of its customer
           SELECT p.order_id, p.items_total, c.customer_state
           FROM per_order AS p
           JOIN orders AS o ON o.order_id = p.order_id
           JOIN customers AS c ON c.customer_id = o.customer_id
       )
       -- one row = one state
       SELECT customer_state, COUNT(*) AS orders, ROUND(AVG(items_total), 2) AS average_items_total
       FROM per_order_with_state
       GROUP BY customer_state
       ORDER BY average_items_total DESC
   """)

,customer_state,orders,average_items_total
0,PB,532,216.67
1,AP,68,198.15
2,AC,81,197.32
3,AL,411,195.41
4,RO,247,186.80
5,PA,970,184.48
6,TO,279,177.86
7,PI,493,176.30
8,MT,903,173.26
9,RN,482,172.27


In [17]:
query("""
      -- ex : a.2
      
    WITH customers_per_zip AS (
        SELECT customer_zip_code_prefix AS zip, COUNT(*) AS n
        FROM customers
        GROUP BY customer_zip_code_prefix
    ),
    geolocation_per_zip AS (
        SELECT geolocation_zip_code_prefix AS zip, COUNT(*) AS n
        FROM geolocation
        GROUP BY geolocation_zip_code_prefix
    )
    SELECT SUM(c.n * g.n) AS rows_the_rule_gives
    FROM customers_per_zip AS c
    JOIN geolocation_per_zip AS g ON g.zip = c.zip
""")

,rows_the_rule_gives
0,15083455.0


> **Interpretation (a.2):** Multiplying grouped counts by zip code demonstrates the exponential row explosion caused by many-to-many relationships when joining tables without strict parent-child constraints.

In [18]:
query("""
    SELECT COUNT(*) AS orders_count
    FROM orders AS o
    JOIN customers AS c ON c.customer_id = o.customer_id
    WHERE c.customer_state = 'SP'
""")

,orders_count
0,41746


In [19]:
query("""
    SELECT COUNT(*) AS rows_count
    FROM orders AS o
    JOIN customers AS c ON c.customer_id = o.customer_id
    JOIN order_items AS i ON i.order_id = o.order_id
    WHERE c.customer_state = 'SP'
""")

,rows_count
0,47449


In [20]:
query("""
    SELECT COUNT(DISTINCT o.order_id) AS unique_orders
    FROM orders AS o
    JOIN customers AS c ON c.customer_id = o.customer_id
    JOIN order_items AS i ON i.order_id = o.order_id
    WHERE c.customer_state = 'SP'
""")

,unique_orders
0,41375


In [21]:
query("""
      
      --ex:c.2
      
    SELECT ROUND(AVG(EXTRACT(DAY FROM order_delivered_customer_date - order_purchase_timestamp)), 2) AS avg_days
    FROM orders
    WHERE order_delivered_customer_date IS NOT NULL
""")

,avg_days
0,12.09


In [22]:
query("""
      --ex:c.2
    SELECT ROUND(AVG(EXTRACT(DAY FROM o.order_delivered_customer_date - o.order_purchase_timestamp)), 2) AS avg_days
    FROM orders AS o
    JOIN order_items AS i ON i.order_id = o.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
""")

,avg_days
0,12.01


> **Interpretation (c.2):** Calculating average delivery duration with and without joining order items shows minor variations, reflecting how multi-item orders influence row-level versus order-level averages.

In [23]:
   query("SELECT 1 = 1 AS normal_test, NULL = NULL AS null_equals_null, NULL <> 'a' AS null_not_a")

,normal_test,null_equals_null,null_not_a
0,True,None,None


In [24]:
query("""
       SELECT
           (SELECT COUNT(*) FROM products) AS all_products,
           (SELECT COUNT(*) FROM products WHERE product_category_name = 'cama_mesa_banho') AS bed_bath_table,
           (SELECT COUNT(*) FROM products WHERE product_category_name <> 'cama_mesa_banho') AS not_bed_bath_table
   """)

,all_products,bed_bath_table,not_bed_bath_table
0,32951,3029,29312


In [25]:
   query("SELECT COUNT(*) FROM products WHERE product_category_name IS DISTINCT FROM 'cama_mesa_banho'")

,count
0,29922


In [26]:
   query("""
       SELECT COUNT(*) AS categories_with_no_2016_sales
       FROM category_translation
       WHERE product_category_name NOT IN (
           SELECT p.product_category_name
           FROM order_items AS i
           JOIN orders AS o ON o.order_id = i.order_id
           JOIN products AS p ON p.product_id = i.product_id
           WHERE o.order_purchase_timestamp < '2017-01-01'
       )
   """)

,categories_with_no_2016_sales
0,0


In [27]:
   query("""
       SELECT COUNT(*) AS categories_with_no_2016_sales
       FROM category_translation AS t
       WHERE NOT EXISTS (
           SELECT 1
           FROM order_items AS i
           JOIN orders AS o ON o.order_id = i.order_id
           JOIN products AS p ON p.product_id = i.product_id
           WHERE o.order_purchase_timestamp < '2017-01-01'
             AND p.product_category_name = t.product_category_name
       )
   """)

,categories_with_no_2016_sales
0,40


In [28]:
   query("""
       SELECT
           ROUND(100 * AVG(CASE
               WHEN order_delivered_customer_date >  order_estimated_delivery_date THEN 1.0
               WHEN order_delivered_customer_date <= order_estimated_delivery_date THEN 0.0
           END), 2) AS late_pct_a,
           ROUND(100 * AVG(CASE
               WHEN order_delivered_customer_date >  order_estimated_delivery_date THEN 1.0
               ELSE 0.0
           END), 2) AS late_pct_b
       FROM orders
   """)

,late_pct_a,late_pct_b
0,8.11,7.87


In [29]:
   query("""
       SELECT
           COUNT(*) AS arrived_orders,
           COUNT(*) FILTER (WHERE order_delivered_customer_date > order_estimated_delivery_date) AS late_orders,
           ROUND(100.0 * COUNT(*) FILTER (WHERE order_delivered_customer_date > order_estimated_delivery_date) / COUNT(*), 2) AS late_pct
       FROM orders
       WHERE order_delivered_customer_date IS NOT NULL
   """)

,arrived_orders,late_orders,late_pct
0,96476,7827,8.11


In [30]:
   query("""
       SELECT
           order_purchase_timestamp,
           date_trunc('day',   order_purchase_timestamp) AS day_start,
           date_trunc('week',  order_purchase_timestamp) AS week_start,
           date_trunc('month', order_purchase_timestamp) AS month_start
       FROM orders
       WHERE order_id = '81b1fd6b50d083d07912773bb6c8095c'
   """)

,order_purchase_timestamp,day_start,week_start,month_start
0,2017-11-24 23:00:33,2017-11-24,2017-11-20,2017-11-01


In [31]:
  query("""
       SELECT date_trunc('week', order_purchase_timestamp)::date AS week_start, COUNT(*) AS orders
       FROM orders
       WHERE order_purchase_timestamp >= '2017-10-30'
         AND order_purchase_timestamp <  '2017-12-11'
       GROUP BY week_start
       ORDER BY week_start
   """)

,week_start,orders
0,2017-10-30,942
1,2017-11-06,1217
2,2017-11-13,1313
3,2017-11-20,3008
4,2017-11-27,2098
5,2017-12-04,1692


In [32]:
 query("""
       SELECT order_id, order_delivered_customer_date, order_estimated_delivery_date
       FROM orders
       WHERE order_delivered_customer_date::date = order_estimated_delivery_date::date
       LIMIT 5
   """)

,order_id,order_delivered_customer_date,order_estimated_delivery_date
0,8563039e855156e48fccee4d611a3196,2018-03-20 00:59:25,2018-03-20
1,1d067305b599c1e0dceb3864056ea527,2018-03-09 21:52:36,2018-03-09
2,c82a457d646f77c8f151e12a3e517ed2,2018-08-15 15:02:09,2018-08-15
3,2c3a5e5f5bc3ae78ed5748461e62d47d,2018-02-06 15:49:01,2018-02-06
4,6cc4eb51469dec50a6130fe1cd07b7f9,2017-05-19 15:55:47,2017-05-19


In [33]:
   query("""
       SELECT
           COUNT(*) FILTER (WHERE order_delivered_customer_date > order_estimated_delivery_date) AS late_by_timestamp,
           COUNT(*) FILTER (WHERE order_delivered_customer_date::date > order_estimated_delivery_date::date) AS late_by_date,
           COUNT(*) FILTER (WHERE order_delivered_customer_date::date = order_estimated_delivery_date::date) AS arrived_on_the_day
       FROM orders
       WHERE order_status = 'delivered'
   """)

,late_by_timestamp,late_by_date,arrived_on_the_day
0,7826,6534,1292


In [34]:
query("""
      
      --ex : a.4
    SELECT 
        date_trunc('month', order_purchase_timestamp)::date AS month_start, 
        COUNT(*) AS orders_count
    FROM orders
    WHERE order_purchase_timestamp >= '2017-01-01'
      AND order_purchase_timestamp < '2018-01-01'
    GROUP BY month_start
    ORDER BY month_start
""")

,month_start,orders_count
0,2017-01-01,800
1,2017-02-01,1780
2,2017-03-01,2682
3,2017-04-01,2404
4,2017-05-01,3700
5,2017-06-01,3245
6,2017-07-01,4026
7,2017-08-01,4331
8,2017-09-01,4285
9,2017-10-01,4631


> **Interpretation:** Monthly order volumes show a steady upward trend throughout 2017, peaking sharply toward November (Black Friday season).

In [35]:
query("""
      
      
      --ex:b.4
    SELECT
        COUNT(*) AS arrived_orders,
        COUNT(*) FILTER (WHERE order_delivered_customer_date::date > order_estimated_delivery_date::date) AS late_orders,
        ROUND(100.0 * COUNT(*) FILTER (WHERE order_delivered_customer_date::date > order_estimated_delivery_date::date) / COUNT(*), 2) AS late_pct_by_date
    FROM orders
    WHERE order_delivered_customer_date IS NOT NULL
""")

,arrived_orders,late_orders,late_pct_by_date
0,96476,6535,6.77


> **Interpretation (b.4):** Evaluating late delivery percentages strictly by calendar dates (ignoring precise timestamps) reduces the overall late rate, as orders arriving on the exact estimated day are classified as on-time.

In [36]:
  query("""
       SELECT column_name, data_type
       FROM information_schema.columns
       WHERE table_name = 'orders' AND column_name = 'order_purchase_timestamp'
   """)

,column_name,data_type
0,order_purchase_timestamp,timestamp without time zone


In [37]:
   query("""
       SELECT
           COUNT(*) FILTER (WHERE EXTRACT(HOUR FROM order_purchase_timestamp) BETWEEN 3 AND 5) AS from_03_to_06,
           COUNT(*) FILTER (WHERE EXTRACT(HOUR FROM order_purchase_timestamp) BETWEEN 6 AND 8) AS from_06_to_09
       FROM orders
   """)

,from_03_to_06,from_06_to_09
0,666,4700


In [38]:
   query("""
       SELECT
           order_purchase_timestamp AS sao_paulo_time,
           (order_purchase_timestamp AT TIME ZONE 'America/Sao_Paulo') AT TIME ZONE 'UTC' AS utc_time
       FROM orders
       WHERE order_id = '81b1fd6b50d083d07912773bb6c8095c'
   """)

,sao_paulo_time,utc_time
0,2017-11-24 23:00:33,2017-11-25 01:00:33


In [39]:
   query("""
       WITH times AS (
           SELECT
               order_purchase_timestamp AS sao_paulo_time,
               (order_purchase_timestamp AT TIME ZONE 'America/Sao_Paulo') AT TIME ZONE 'UTC' AS utc_time
           FROM orders
       )
       SELECT
           COUNT(*) FILTER (WHERE sao_paulo_time::date = '2017-11-24') AS by_sao_paulo_date,
           COUNT(*) FILTER (WHERE utc_time::date = '2017-11-24') AS by_utc_date
       FROM times
   """)

,by_sao_paulo_date,by_utc_date
0,1176,1068


In [40]:
   query("""
       SELECT
           sao_paulo_time,
           (sao_paulo_time AT TIME ZONE 'America/Sao_Paulo') AT TIME ZONE 'UTC' AS utc_time
       FROM (VALUES
           (TIMESTAMP '2017-07-10 23:30'),
           (TIMESTAMP '2017-11-24 23:30')
       ) AS examples (sao_paulo_time)
   """)

,sao_paulo_time,utc_time
0,2017-07-10 23:30:00,2017-07-11 02:30:00
1,2017-11-24 23:30:00,2017-11-25 01:30:00


In [41]:
query("""
      --ex: a.5
    WITH times AS (
        SELECT
            order_purchase_timestamp AS sao_paulo_time,
            (order_purchase_timestamp AT TIME ZONE 'America/Sao_Paulo') AT TIME ZONE 'UTC' AS utc_time
        FROM orders
    )
    SELECT
        COUNT(*) FILTER (WHERE sao_paulo_time::date <> utc_time::date) AS different_date_orders,
        ROUND(100.0 * COUNT(*) FILTER (WHERE sao_paulo_time::date <> utc_time::date) / COUNT(*), 1) AS diff_pct
    FROM times
""")

,different_date_orders,diff_pct
0,14377,14.5


> **Interpretation (a.5):** Comparing local São Paulo timestamps with UTC equivalents shows that roughly 14.5% of orders shift calendar dates due to late-night transaction timing.

In [42]:
query("""
      
      --ex : b.5
      
    WITH times AS (
        SELECT
            order_purchase_timestamp AS sao_paulo_time,
            (order_purchase_timestamp AT TIME ZONE 'America/Sao_Paulo') AT TIME ZONE 'UTC' AS utc_time
        FROM orders
    )
    SELECT
        COUNT(*) FILTER (WHERE sao_paulo_time >= TIMESTAMP '2017-11-01' AND sao_paulo_time < TIMESTAMP '2017-12-01') AS sp_nov,
        COUNT(*) FILTER (WHERE utc_time >= TIMESTAMP '2017-11-01' AND utc_time < TIMESTAMP '2017-12-01') AS utc_nov
    FROM times
""")

,sp_nov,utc_nov
0,7544,7529


> **Interpretation:** Shifting timestamps between local São Paulo time and UTC alters the order count for November due to late-night transactions crossing calendar boundaries.